# Option 1 + 4 on DocLens (Qwen3-VL-8B): unanimity gate + visual Adjudicator

Runs **on top of the finished DocLens run** (200 questions, 56.3%). Nothing of DocLens is recomputed:
the Page Navigator outputs and the 8 sampled answers (with their scores) are read from
`doclens_cdc/results/qwen_doclens_original.jsonl`, which comes with the code.

* **Option 1 (unanimity gate):** all 8 sampled answers agree -> return that answer, **no Adjudicator call** (96 questions).
* **Option 4 (visual Adjudicator):** answers disagree -> the Adjudicator sees the **same evidence pages** the
  Answer Sampler saw (screenshots + zoomed figures/tables + text) plus the 8 analyses/answers, and picks one (104 questions).

Compared with B1 = DocLens (56.3), B2 = majority vote (54.8) and Option 1 alone, on the same questions and the same scoring.

**How to run**
1. *Settings*: **GPU T4 x2**, **Internet On**.
2. *Add Input* -> *Your Work* -> **Agentic** (the version with `parsed_MMLongBenchDoc.tar`), exactly as for the DocLens run.
3. **Save Version -> "Save & Run All (Commit)" -> Save.** Expected time: about 2-4 hours (setup ~40 min + 104 Qwen calls).
4. When it is *Complete*: *Output* -> `results_qwen/MMLongBenchDoc/visual_adj_report.md` (also shown at the end of the notebook).
   If it stopped before all 200 questions were done: add this notebook's own latest version as an input too and run again; it continues.


In [ ]:
# ======================= SETTINGS =======================
import time
T_START = time.time()
TIME_BUDGET_H = 11.0              # stop the runs after this many hours (Kaggle ends every session at 12)
WORK = "/kaggle/working"
CDC_REPO = "https://github.com/Mariam1543/aqqal123.git"
CDC_BRANCH = "claude/wizardly-rubin-kdthhy"
DATASET = "MMLongBenchDoc"

N_SAMPLE = 200                    # Table 9 uses a random sample of 200 questions (paper, Appendix C)
PART = "AB"                        # "A" = a random half (100), "B" = the other half (100), "AB" = all 200.
                                  # Two accounts: one runs "A", the other "B"; then one run with "AB" + both
                                  # outputs attached merges them into the full 200 (nothing is recomputed).
SEED = 0
RUN_ONLY = ["qwen_doclens"]      # only its settings are used (model name, k = 8); DocLens is NOT re-run

MAX_MODEL_LEN = 40960             # context that fits next to the 8B weights on 2x T4
NAV_CHUNK = 6                     # Page Navigator pages per request (short prompts: attention on T4 grows with length^2)
QWEN_PARALLEL = 4                 # Qwen requests on the server at once (each carries all 8 samples)
MAX_IMAGES = 48                   # images per request accepted by the server
MAX_CONCURRENT = 4
PAPER_PROMPTS = True
PILOT_QUESTIONS = 0               # 0 = no pilot (already measured: ~4.5 min per DocLens question)

## 1. Get the code

In [ ]:
import os, sys, subprocess, shutil, json, glob, random, time
os.chdir(WORK)
if not os.path.exists("DocLens"):
    subprocess.run(["git", "clone", "https://github.com/dwzhu-pku/DocLens.git"], check=True)
shutil.rmtree("doclens_cdc", ignore_errors=True)   # always take the latest version of the extension
shutil.rmtree("_cdc_repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", CDC_BRANCH, CDC_REPO, "_cdc_repo"], check=True)
shutil.copytree("_cdc_repo/doclens_cdc", "doclens_cdc")
shutil.rmtree("_cdc_repo")
DOCLENS, CDC = f"{WORK}/DocLens", f"{WORK}/doclens_cdc"
ROOT = f"{WORK}/full_{DATASET}"
RESULTS = f"{WORK}/results_qwen/{DATASET}"
os.makedirs(RESULTS, exist_ok=True)
sys.path.insert(0, CDC)
from reproduction.experiments import EXPERIMENTS, run_args
SELECTED = {k: EXPERIMENTS[k] for k in RUN_ONLY}
for k, e in SELECTED.items():
    print(f"{k:14s} {e['note']:40s} K={e['k1']}/{e['k2']}")

## 2. Install vLLM and the DocLens dependencies (about 10 minutes)

In [ ]:
!pip install -q "vllm>=0.11.0" 2>&1 | tail -n 3
!pip install -q -r {CDC}/requirements-kaggle.txt 2>&1 | tail -n 2
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())"

## 3. Restore the MinerU results and re-create the page images

In [ ]:
tars = glob.glob("/kaggle/input/**/parsed_MMLongBenchDoc.tar", recursive=True)
assert tars, "Add the saved notebook version containing parsed_MMLongBenchDoc.tar as an input (right panel: Add Input)."
if not os.path.exists(f"{ROOT}/clean/data/{DATASET}/documents"):
    !tar -xf {tars[0]} -C {WORK}
!python {CDC}/scripts/download_data.py --dataset {DATASET} --out {DOCLENS}/data
SAMPLES = f"{DOCLENS}/data/{DATASET}/samples.json"
DOCUMENTS = f"{DOCLENS}/data/{DATASET}/documents"
# The tar holds MinerU's output but not the page screenshots; re-render them (skips existing ones).
!python {CDC}/scripts/prepare_subset.py --all --samples {SAMPLES} --documents {DOCUMENTS} --out_root {ROOT} --dataset_name {DATASET} --workers 4
docs = f"{ROOT}/clean/data/{DATASET}/documents"
def n_done(path, want=None):
    # finished questions in a results file (failed attempts and repeats not counted; only `want` if given)
    if not os.path.exists(path):
        return 0
    keys = set()
    for line in open(path, encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            if "cdc_error" not in r:
                keys.add((r["doc_id"], r["question"], str(r.get("answer", ""))))
    return len(keys & want) if want is not None else len(keys)

# Resume / merge: add the finished questions of every attached earlier output (this notebook's own versions,
# or the other account's part) to the local results files
def qkey(r):
    return (r["doc_id"], r["question"], str(r.get("answer", "")))

for prev in sorted(glob.glob(f"/kaggle/input/**/results_qwen/{DATASET}/*.jsonl", recursive=True)):
    dst = f"{RESULTS}/{os.path.basename(prev)}"
    have = set()
    if os.path.exists(dst):
        have = {qkey(json.loads(l)) for l in open(dst, encoding="utf-8") if l.strip()
                and "cdc_error" not in json.loads(l)}
    added = 0
    with open(dst, "a", encoding="utf-8") as f:
        for line in open(prev, encoding="utf-8"):
            if not line.strip():
                continue
            r = json.loads(line)
            if "cdc_error" in r or qkey(r) in have:
                continue
            f.write(line.rstrip("\n") + "\n"); have.add(qkey(r)); added += 1
    print(f"Merged {os.path.basename(prev)} from {os.path.dirname(prev)}: +{added} questions")
print(len(glob.glob(f"{docs}/*/*.jpeg")), "page images,", len(glob.glob(f"{docs}/*/MinerU_Page*/*.md")), "parsed pages (expected 6529 each)")

In [ ]:
# A seeded random sample of N_SAMPLE questions (like Table 9), in random order; part A = its first half,
# part B = its second half. The questions are written in the benchmark's order.
all_q = json.load(open(SAMPLES))
order = random.Random(SEED).sample(range(len(all_q)), N_SAMPLE)
half = N_SAMPLE // 2
picked = set({"A": order[:half], "B": order[half:], "AB": order}[PART])
N_QUESTIONS = len(picked)
QUESTIONS = f"{ROOT}/samples_random{N_SAMPLE}_seed{SEED}_part{PART}.json"
json.dump([q for i, q in enumerate(all_q) if i in picked], open(QUESTIONS, "w"), ensure_ascii=False, indent=1)
WANT = {(q["doc_id"], q["question"], str(q.get("answer", ""))) for i, q in enumerate(all_q) if i in picked}
print(f"part {PART}: {N_QUESTIONS} questions from {len({all_q[i]['doc_id'] for i in picked})} documents -> {QUESTIONS}")

## 4. Start the Qwen3-VL-8B server (downloads ~17 GB the first time, then loads onto both T4s)

In [ ]:
import urllib.request
log = open(f"{WORK}/vllm.log", "w")
server = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen3-VL-8B-Instruct",
    "--tensor-parallel-size", "2", "--dtype", "half",
    "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.92",
    "--limit-mm-per-prompt", json.dumps({"image": MAX_IMAGES}),
    "--mm-processor-kwargs", json.dumps({"max_pixels": 1003520}),
    "--max-num-seqs", "32", "--enable-prefix-caching", "--enforce-eager",
    "--disable-custom-all-reduce", "--port", "8000",
], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)  # survives a Stop/Interrupt

start = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        print(f"Server ready after {(time.time() - start) / 60:.1f} min"); break
    except Exception:
        if server.poll() is not None or time.time() - start > 40 * 60:
            print(open(f"{WORK}/vllm.log").read()[-6000:])
            raise RuntimeError("vLLM did not start: send the log above.")
        time.sleep(20)

In [ ]:
# Smoke test: one real question about one real page
import base64, openai
client = openai.OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
page = sorted(glob.glob(f"{docs}/*/1.jpeg"))[0]
b64 = base64.b64encode(open(page, "rb").read()).decode()
r = client.chat.completions.create(model="Qwen/Qwen3-VL-8B-Instruct", max_tokens=60, messages=[{"role": "user", "content": [
    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
    {"type": "text", "text": "What is the title of this page?"}]}])
print(page, "->", r.choices[0].message.content)

## 5. Run settings

In [ ]:
common = (f"python {CDC}/run_cdc.py --doclens_root {DOCLENS} --data_root {ROOT}/clean --variant clean "
          f"--dataset_name {DATASET} --samples {QUESTIONS} --max_concurrent {MAX_CONCURRENT} "
          f"--qwen_base_url http://localhost:8000/v1 --qwen_max_images {MAX_IMAGES} --nav_chunk_pages {NAV_CHUNK} --qwen_parallel_requests {QWEN_PARALLEL} "
          f"{'--paper_prompts' if PAPER_PROMPTS else ''}")


## 6. Option 1 + 4 run
Progress is printed every 10 minutes; the log is `results_qwen/MMLongBenchDoc/qwen_visual_adj.log`.

In [ ]:
# Option 1 + 4 on all 200 questions (resumes from saved results; stops at TIME_BUDGET_H)
import signal
DOCLENS_RUN = f"{CDC}/results/qwen_doclens_original.jsonl"     # the finished DocLens run (56.3%)
assert os.path.exists(DOCLENS_RUN), "results/qwen_doclens_original.jsonl is missing from the code branch"
print(n_done(DOCLENS_RUN, WANT), "of", N_QUESTIONS, "questions found in the DocLens run")
out = f"{RESULTS}/qwen_visual_adj.jsonl"
cmd = (f"{common} {run_args(SELECTED['qwen_doclens'])} --mode visual_adj --reuse_from {DOCLENS_RUN} --output {out}")
for attempt in range(3):          # a second pass retries questions whose call failed
    if n_done(out, WANT) >= N_QUESTIONS:
        break
    if time.time() > T_START + TIME_BUDGET_H * 3600 - 10 * 60:
        print("Time budget reached; add this version as input and run again to continue."); break
    print(f"pass {attempt + 1}: {n_done(out, WANT)}/{N_QUESTIONS} done", flush=True)
    log = open(f"{RESULTS}/qwen_visual_adj.log", "a")
    proc = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    t_last = time.time()
    while proc.poll() is None:
        time.sleep(30)
        if time.time() - t_last > 600:
            t_last = time.time()
            print(f"  {time.strftime('%H:%M')}: {n_done(out, WANT)}/{N_QUESTIONS} done", flush=True)
        if time.time() > T_START + TIME_BUDGET_H * 3600:
            os.killpg(proc.pid, signal.SIGTERM); proc.wait()
            print("  stopped at the time budget (saved)", flush=True); break
    log.close()
print(open(f"{RESULTS}/qwen_visual_adj.log").read()[-1500:])
print(f"{n_done(out, WANT)}/{N_QUESTIONS} done")


## 7. Report: Option 1 + 4 vs DocLens, majority vote and Option 1

In [ ]:
!python {CDC}/scripts/report_visual_adj.py {RESULTS}/qwen_visual_adj.jsonl --out {RESULTS}/visual_adj_report.md
from IPython.display import Markdown, display
if os.path.exists(f"{RESULTS}/visual_adj_report.md"):
    display(Markdown(open(f"{RESULTS}/visual_adj_report.md").read()))


## 8. Clean up so the output saves quickly

In [ ]:
subprocess.run(["pkill", "-f", "vllm serve"])
!rm -rf {ROOT} {DOCLENS}/data
!du -sh {WORK}
print("qwen_visual_adj.jsonl:", n_done(f"{RESULTS}/qwen_visual_adj.jsonl", WANT), "of", N_QUESTIONS)
